# The report

This notebook derives every number, figure and sentence of
`reports/faithfulness_report.html` from the cached benchmark outputs of
notebook 06. Nothing is recomputed and no number is typed by hand: the prose
is assembled from the same tables the figures draw, so the report cannot
drift from the data it describes. Re-running notebook 06 with fresh caches
and then this notebook regenerates the report end to end.

## Configuration

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIRECTORY = PROJECT_ROOT / "artifacts"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
REPORT_PATH = REPORTS_DIRECTORY / "faithfulness_report.html"

DATASET_DIRECTORIES = {
    "imdb": ARTIFACTS_DIRECTORY / "sample_2000",
    "yelp": ARTIFACTS_DIRECTORY / "yelp_2000",
    "agnews": ARTIFACTS_DIRECTORY / "agnews_2000",
}
DATASET_TASKS = {
    "imdb": "movie review sentiment",
    "yelp": "business review sentiment",
    "agnews": "news topic, business vs sci/tech",
}

OBSERVATION_COUNT = 100
EVALUATION_BUDGET = 500
PERCENTILES = [20, 40, 60, 80]
CHUNK_SIZES = [1, 4, 8]
COMBINATIONS = ["absolute_sum", "sum_absolute", "max_absolute"]
RANDOM_SEED = 42

SHAP_COLOR = "#2a78d6"
LIME_COLOR = "#eb6834"
OCCLUSION_COLOR = "#1baf7a"
MUTED_COLOR = "#898781"
GRID_COLOR = "#e1e0d9"
AXIS_COLOR = "#c3c2b7"
INK_COLOR = "#0b0b0b"
SECONDARY_INK_COLOR = "#52514e"
SURFACE_COLOR = "#fcfcfb"
PAGE_COLOR = "#f9f9f7"
NEUTRAL_MIDPOINT_COLOR = "#f0efec"
METHOD_COLORS = {"shap": SHAP_COLOR, "lime": LIME_COLOR, "occlusion": OCCLUSION_COLOR}

print(f"Report will be written to {REPORT_PATH}")

Report will be written to /home/user/projects/shapley/reports/faithfulness_report.html


## Load the benchmark outputs

Everything comes from the two parquet files notebook 06 wrote, the checkpoint
metadata, and the per dataset attribution caches. `build_paired_grid` is the
same summary the benchmark notebook displayed, recomputed here from the tidy
table so the report and the notebook can never disagree.

In [2]:
import joblib
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from textfaith.experiment import average_shifts_over_thresholds, build_paired_grid

tidy = pd.read_parquet(ARTIFACTS_DIRECTORY / "faithfulness_results.parquet")
timing = pd.read_parquet(ARTIFACTS_DIRECTORY / "timing_results.parquet").reset_index()
method_comparison = pd.read_parquet(ARTIFACTS_DIRECTORY / "method_comparison.parquet")
metadata_by_dataset = {
    name: json.loads((directory / "metadata.json").read_text())
    for name, directory in DATASET_DIRECTORIES.items()
}
paired_grid = build_paired_grid(tidy, budget=EVALUATION_BUDGET)
average_shifts = average_shifts_over_thresholds(tidy, value_column="relative_shift")

attributions_by_dataset = {}
for dataset_name, directory in DATASET_DIRECTORIES.items():
    attributions_by_dataset[dataset_name] = {
        "shap": joblib.load(
            directory
            / f"shap_attributions_{OBSERVATION_COUNT}obs_{EVALUATION_BUDGET}evals.joblib"
        )["attributions"],
        "lime": joblib.load(
            directory
            / f"lime_attributions_{OBSERVATION_COUNT}obs_{EVALUATION_BUDGET}samples.joblib"
        )["attributions"],
        "occlusion": joblib.load(
            directory / f"occlusion_attributions_{OBSERVATION_COUNT}obs.joblib"
        )["attributions"],
    }


def median_rank_correlation(first_attributions, second_attributions):
    return float(
        np.median(
            [
                spearmanr(first, second).statistic
                for first, second in zip(first_attributions, second_attributions)
            ]
        )
    )


rank_correlation_rows = []
for dataset_name, method_attributions in attributions_by_dataset.items():
    rank_correlation_rows.append(
        {
            "dataset": dataset_name,
            "shap_vs_lime": median_rank_correlation(
                method_attributions["shap"], method_attributions["lime"]
            ),
            "shap_vs_occlusion": median_rank_correlation(
                method_attributions["shap"], method_attributions["occlusion"]
            ),
            "lime_vs_occlusion": median_rank_correlation(
                method_attributions["lime"], method_attributions["occlusion"]
            ),
        }
    )
rank_correlations = pd.DataFrame(rank_correlation_rows).set_index("dataset")

print(f"{len(tidy)} tidy rows, {len(paired_grid)} grid conditions")
print(rank_correlations.round(2).to_string())

/home/user/projects/shapley/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


216000 tidy rows, 27 grid conditions
         shap_vs_lime  shap_vs_occlusion  lime_vs_occlusion
dataset                                                    
imdb             0.23               0.21               0.28
yelp             0.37               0.17               0.29
agnews           0.75               0.46               0.58


## Figures

Colors follow a validated categorical palette, SHAP in blue and LIME in
orange throughout, controls dashed in the method's own hue so identity is
never lost, and the advantage matrix uses a blue to orange diverging scale
through a neutral gray midpoint so the sign reads as which method it favors.

In [3]:
import base64
import io

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE_COLOR,
        "axes.facecolor": SURFACE_COLOR,
        "savefig.facecolor": SURFACE_COLOR,
        "axes.edgecolor": AXIS_COLOR,
        "axes.labelcolor": SECONDARY_INK_COLOR,
        "text.color": INK_COLOR,
        "xtick.color": SECONDARY_INK_COLOR,
        "ytick.color": SECONDARY_INK_COLOR,
        "grid.color": GRID_COLOR,
        "font.size": 11,
        "savefig.dpi": 150,
        "savefig.bbox": "tight",
    }
)

report_images = {}


def figure_to_data_uri(figure) -> str:
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png")
    plt.close(figure)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


print("figure helpers ready")

figure helpers ready


In [4]:
CURVE_CHUNK_SIZE = 4
CURVE_COMBINATION = "absolute_sum"

figure, axes = plt.subplots(2, 3, figsize=(12.5, 7.0), sharey="row", sharex=True)
for column_index, dataset_name in enumerate(DATASET_DIRECTORIES):
    for row_index, direction in enumerate(["sufficiency", "comprehensiveness"]):
        axis = axes[row_index][column_index]
        for method in ["shap", "lime"]:
            for source, style, line_width, alpha in [
                (f"{method}_{EVALUATION_BUDGET}", "-o", 2.0, 1.0),
                (f"{method}_{EVALUATION_BUDGET}_random", "--", 1.5, 0.55),
            ]:
                subset = tidy[
                    (tidy["dataset"] == dataset_name)
                    & (tidy["source"] == source)
                    & (tidy["chunk_size"] == CURVE_CHUNK_SIZE)
                    & (tidy["combination"] == CURVE_COMBINATION)
                    & (tidy["direction"] == direction)
                ]
                means = subset.groupby("percentile")["relative_shift"].mean() * 100
                axis.plot(
                    means.index, means.to_numpy(), style,
                    color=METHOD_COLORS[method], linewidth=line_width,
                    markersize=6, alpha=alpha,
                    label=method.upper() if not source.endswith("_random") else None,
                )
        axis.grid(alpha=0.6, linewidth=0.7)
        axis.set_xticks(PERCENTILES)
        for spine_name in ["top", "right"]:
            axis.spines[spine_name].set_visible(False)
        if row_index == 0:
            axis.set_title(dataset_name, fontsize=12)
        else:
            axis.set_xlabel("percent of text cut")

axes[0][0].set_ylabel("sufficiency, % of confidence")
axes[1][0].set_ylabel("comprehensiveness, % of confidence")
axes[0][0].legend(
    title="solid: method, dashed: its shuffled control",
    fontsize=9, title_fontsize=9, frameon=False,
)
figure.suptitle(
    f"Mean shift as percent of the model's confidence, {CURVE_COMBINATION}, chunk size {CURVE_CHUNK_SIZE}",
    y=1.0, fontsize=13,
)
figure.tight_layout()
report_images["curves"] = figure_to_data_uri(figure)
print("curves figure built")

curves figure built


In [5]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

advantage_colormap = LinearSegmentedColormap.from_list(
    "advantage", [LIME_COLOR, NEUTRAL_MIDPOINT_COLOR, SHAP_COLOR]
)
column_keys = [
    (chunk_size, combination)
    for chunk_size in CHUNK_SIZES
    for combination in COMBINATIONS
]
dataset_names = list(DATASET_DIRECTORIES)

maximum_advantage = max(
    paired_grid["sufficiency_shap_advantage"].abs().max(),
    paired_grid["comprehensiveness_shap_advantage"].abs().max(),
)
normalize = Normalize(-maximum_advantage, maximum_advantage)

figure, axes = plt.subplots(2, 1, figsize=(12.5, 5.8))
for axis, direction in zip(axes, ["sufficiency", "comprehensiveness"]):
    matrix = np.array(
        [
            [
                paired_grid.loc[
                    (dataset_name, chunk_size, combination),
                    f"{direction}_shap_advantage",
                ]
                for chunk_size, combination in column_keys
            ]
            for dataset_name in dataset_names
        ]
    )
    axis.imshow(matrix, cmap=advantage_colormap, norm=normalize, aspect="auto")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix[row, column]
            red, green, blue, _ = advantage_colormap(normalize(value))
            luminance = 0.299 * red + 0.587 * green + 0.114 * blue
            axis.text(
                column, row, f"{value:+.1f}", ha="center", va="center", fontsize=9,
                fontweight="bold" if abs(value) > 2 else "normal",
                color="white" if luminance < 0.5 else INK_COLOR,
            )
    axis.set_yticks(range(len(dataset_names)), dataset_names)
    axis.set_xticks(
        range(len(column_keys)),
        [f"chunk {chunk_size}\n{combination}" for chunk_size, combination in column_keys],
        fontsize=8,
    )
    axis.set_title(
        f"{direction}: paired advantage in standard errors, "
        "blue favors SHAP, orange favors LIME, bold exceeds noise",
        fontsize=11, loc="left",
    )
    axis.tick_params(length=0)
    for spine in axis.spines.values():
        spine.set_visible(False)

figure.tight_layout()
report_images["advantage"] = figure_to_data_uri(figure)
print("advantage figure built")

advantage figure built


In [6]:
timing_indexed = timing.set_index(["dataset", "method", "budget"])
bar_positions = np.arange(len(dataset_names))
bar_width = 0.16

SOURCE_BAR_SPECS = [
    ("shap", "500", SHAP_COLOR, 1.0, "SHAP 500"),
    ("shap", "50", SHAP_COLOR, 0.45, "SHAP 50"),
    ("lime", "500", LIME_COLOR, 1.0, "LIME 500"),
    ("lime", "50", LIME_COLOR, 0.45, "LIME 50"),
    ("occlusion", "word_count", OCCLUSION_COLOR, 1.0, "occlusion"),
]

figure, axis = plt.subplots(figsize=(9.0, 4.2))
for offset_index, (method, budget, color, alpha, label) in enumerate(SOURCE_BAR_SPECS):
    seconds = [
        timing_indexed.loc[(dataset_name, method, budget), "seconds_per_review"]
        for dataset_name in dataset_names
    ]
    axis.bar(
        bar_positions + (offset_index - 2) * bar_width, seconds, bar_width * 0.92,
        color=color, alpha=alpha, label=label,
        edgecolor=SURFACE_COLOR, linewidth=1.5,
    )

axis.set_xticks(bar_positions, dataset_names)
axis.set_ylabel("seconds per explained review")
axis.set_title(
    "Attribution cost per source, 8 core CPU (values in Table 2)",
    fontsize=12, loc="left",
)
axis.grid(axis="y", alpha=0.6, linewidth=0.7)
axis.set_axisbelow(True)
for spine_name in ["top", "right"]:
    axis.spines[spine_name].set_visible(False)
axis.legend(frameon=False, fontsize=9, ncols=5)
figure.tight_layout()
report_images["timing"] = figure_to_data_uri(figure)
print("timing figure built")

timing figure built


In [7]:
BUDGET_CHUNK_SIZE = 1
BUDGET_COMBINATION = "absolute_sum"

predictions_per_review = {
    (row.dataset, row.method, row.budget): row.prediction_count / OBSERVATION_COUNT
    for row in timing.itertuples()
}

figure, axes = plt.subplots(2, 3, figsize=(12.5, 7.0), sharey="row", sharex=True)
for column_index, dataset_name in enumerate(dataset_names):
    for row_index, direction in enumerate(["sufficiency", "comprehensiveness"]):
        axis = axes[row_index][column_index]
        subset = average_shifts[
            (average_shifts["dataset"] == dataset_name)
            & (~average_shifts["is_random"])
            & (average_shifts["chunk_size"] == BUDGET_CHUNK_SIZE)
            & (average_shifts["combination"] == BUDGET_COMBINATION)
            & (average_shifts["direction"] == direction)
        ]
        means = subset.groupby(["method", "budget"])["average_shift"].mean()
        labeled = row_index == 0 and column_index == 0
        for method in ["shap", "lime"]:
            budgets = ["50", "500"]
            x_values = [
                predictions_per_review[(dataset_name, method, budget)]
                for budget in budgets
            ]
            y_values = [means[(method, budget)] * 100 for budget in budgets]
            axis.plot(
                x_values, y_values, "-o", color=METHOD_COLORS[method],
                linewidth=2.0, markersize=6,
                label=method.upper() if labeled else None,
            )
        axis.plot(
            [predictions_per_review[(dataset_name, "occlusion", "word_count")]],
            [means[("occlusion", "word_count")] * 100],
            "D", color=METHOD_COLORS["occlusion"], markersize=8,
            label="occlusion" if labeled else None,
        )
        axis.set_xscale("log")
        axis.set_xticks([50, 100, 200, 500], ["50", "100", "200", "500"])
        axis.minorticks_off()
        axis.grid(alpha=0.6, linewidth=0.7)
        for spine_name in ["top", "right"]:
            axis.spines[spine_name].set_visible(False)
        if row_index == 0:
            axis.set_title(dataset_name, fontsize=12)
        else:
            axis.set_xlabel("predictions per review (log scale)")

axes[0][0].set_ylabel("sufficiency, % of confidence")
axes[1][0].set_ylabel("comprehensiveness, % of confidence")
axes[0][0].legend(fontsize=9, frameon=False)
figure.suptitle(
    f"Faithfulness versus evaluation budget, chunk size {BUDGET_CHUNK_SIZE}, "
    f"{BUDGET_COMBINATION}",
    y=1.0, fontsize=13,
)
figure.tight_layout()
report_images["budget"] = figure_to_data_uri(figure)
print("budget figure built")

budget figure built


## Worked example

The same review notebook 03 used, chosen by the same criterion, correctly and
confidently predicted, at least 60 words, shortest such review. Both methods
explain it from the fresh equal budget caches, and both cut renderings use the
identical threshold so the difference between the methods is the only thing
that varies.

In [8]:
from sklearn.model_selection import train_test_split

from textfaith.checkpoint import load_predictor
from textfaith.faithfulness import build_word_masker, select_kept_words
from textfaith.rendering import render_words, word_figure_size, wrap_words

EXAMPLE_MINIMUM_WORD_COUNT = 60
EXAMPLE_MINIMUM_CONFIDENCE = 0.80
EXAMPLE_CHUNK_SIZE = 4
EXAMPLE_PERCENTILE = 40
EXAMPLE_CHARACTERS_PER_LINE = 76

predict_probabilities, _ = load_predictor(DATASET_DIRECTORIES["imdb"])
test_reviews = pd.read_parquet(DATASET_DIRECTORIES["imdb"] / "test.parquet")
sampled_reviews, _ = train_test_split(
    test_reviews, train_size=OBSERVATION_COUNT,
    random_state=RANDOM_SEED, stratify=test_reviews["label"],
)
texts = sampled_reviews["text"].tolist()
labels = sampled_reviews["label"].tolist()
masker = build_word_masker()
imdb_attributions = attributions_by_dataset["imdb"]

all_probabilities = predict_probabilities(texts)
predicted_classes = all_probabilities.argmax(axis=1)
confidences = all_probabilities.max(axis=1)
word_counts = np.array([len(attribution) for attribution in imdb_attributions["shap"]])
candidates = (
    (predicted_classes == np.array(labels))
    & (confidences >= EXAMPLE_MINIMUM_CONFIDENCE)
    & (word_counts >= EXAMPLE_MINIMUM_WORD_COUNT)
)
assert candidates.any(), "No review satisfies the example criterion"
candidate_indices = np.flatnonzero(candidates)
chosen_index = int(candidate_indices[np.argmin(word_counts[candidates])])

chosen_text = texts[chosen_index]
chosen_words = masker.data_transform(chosen_text)[0]
predicted_class = int(predicted_classes[chosen_index])
class_name = {0: "negative", 1: "positive"}[predicted_class]
uncut_probability = float(all_probabilities[chosen_index, predicted_class])
example_rank_correlation = float(
    spearmanr(
        imdb_attributions["shap"][chosen_index], imdb_attributions["lime"][chosen_index]
    ).statistic
)

line_count = wrap_words(chosen_words, EXAMPLE_CHARACTERS_PER_LINE)[-1][1] + 1
panel_width, panel_height = word_figure_size(
    EXAMPLE_CHARACTERS_PER_LINE, line_count, 10
)

EXAMPLE_METHODS = ["shap", "lime", "occlusion"]

figure, axes = plt.subplots(
    len(EXAMPLE_METHODS), 1,
    figsize=(panel_width, (panel_height + 0.5) * len(EXAMPLE_METHODS)),
)
for axis, method in zip(axes, EXAMPLE_METHODS):
    render_words(
        axis, chosen_words, imdb_attributions[method][chosen_index],
        characters_per_line=EXAMPLE_CHARACTERS_PER_LINE, fontsize=10,
    )
    axis.set_title(
        f"{method.upper()} word attributions toward {class_name}",
        fontsize=11, loc="left",
    )
figure.tight_layout()
report_images["example_maps"] = figure_to_data_uri(figure)

cut_masks = {}
cut_texts = {}
for method in EXAMPLE_METHODS:
    kept_words = select_kept_words(
        imdb_attributions[method][chosen_index],
        EXAMPLE_PERCENTILE, EXAMPLE_CHUNK_SIZE, "absolute_sum",
    )
    cut_masks[method] = ~kept_words
    cut_texts[method] = masker(kept_words, chosen_text)[0][0]
example_cut_probabilities = predict_probabilities(
    [cut_texts[method] for method in EXAMPLE_METHODS]
)

figure, axes = plt.subplots(
    len(EXAMPLE_METHODS), 1,
    figsize=(panel_width, (panel_height + 0.5) * len(EXAMPLE_METHODS)),
)
for axis, method, probabilities in zip(
    axes, EXAMPLE_METHODS, example_cut_probabilities
):
    kept_count = int((~cut_masks[method]).sum())
    render_words(
        axis, chosen_words, imdb_attributions[method][chosen_index],
        cut_mask=cut_masks[method],
        characters_per_line=EXAMPLE_CHARACTERS_PER_LINE, fontsize=10,
    )
    axis.set_title(
        f"{method.upper()} cut {EXAMPLE_PERCENTILE}% at chunk size {EXAMPLE_CHUNK_SIZE}: "
        f"kept {kept_count}/{len(chosen_words)} words, "
        f"P({class_name}) = {float(probabilities[predicted_class]):.3f}",
        fontsize=11, loc="left",
    )
figure.tight_layout()
report_images["example_cuts"] = figure_to_data_uri(figure)

print(
    f"example review: position {chosen_index}, {len(chosen_words)} words, "
    f"uncut P({class_name}) = {uncut_probability:.3f}, "
    f"SHAP to LIME rank correlation {example_rank_correlation:.2f}"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21064.14it/s]

example review: position 76, 64 words, uncut P(positive) = 0.853, SHAP to LIME rank correlation 0.69


## Headline statistics for the prose

Every sentence in the report that contains a number gets it from here.

In [9]:
verdict_counts = {}
for direction in ["sufficiency", "comprehensiveness"]:
    advantages = paired_grid[f"{direction}_shap_advantage"]
    verdict_counts[direction] = {
        "shap": int((advantages > 2).sum()),
        "lime": int((advantages < -2).sum()),
        "noise": int((advantages.abs() <= 2).sum()),
        "total": int(len(advantages)),
    }

control_columns = [column for column in paired_grid.columns if "vs_control" in column]
weakest_control_advantage = float(paired_grid[control_columns].min().min())
control_total = int(paired_grid[control_columns].size)
control_significant = int((paired_grid[control_columns] > 2).sum().sum())
assert (paired_grid[control_columns] > 0).all().all(), (
    "A method fell below its shuffled control somewhere, the prose must change"
)

SOURCE_PAIRS = [
    ("shap_500", "lime_500"),
    ("shap_50", "lime_50"),
    ("occlusion", "shap_500"),
    ("occlusion", "lime_500"),
    ("shap_500", "shap_50"),
    ("lime_500", "lime_50"),
]

comparison_summary_rows = []
for first, second in SOURCE_PAIRS:
    subset = method_comparison.loc[f"{first} vs {second}"]
    survivors = subset[subset["holm_significant"]]
    comparison_summary_rows.append(
        {
            "comparison": f"{first} vs {second}",
            "conditions": int(len(subset)),
            "significant_after_holm": int(len(survivors)),
            "favoring_first": int((survivors["first_advantage"] > 0).sum()),
            "favoring_second": int((survivors["first_advantage"] < 0).sum()),
        }
    )
comparison_summary = pd.DataFrame(comparison_summary_rows).set_index("comparison")

assert comparison_summary.loc["occlusion vs shap_500", "favoring_first"] == 0, (
    "occlusion beat SHAP somewhere, the 'never wins' prose must change"
)
assert comparison_summary.loc["occlusion vs lime_500", "favoring_first"] == 0, (
    "occlusion beat LIME somewhere, the 'never wins' prose must change"
)
assert comparison_summary.loc["shap_500 vs shap_50", "favoring_second"] == 0, (
    "the cheap SHAP arm won somewhere, the budget prose must change"
)
assert comparison_summary.loc["lime_500 vs lime_50", "favoring_second"] == 0, (
    "the cheap LIME arm won somewhere, the budget prose must change"
)

head_to_head = method_comparison.loc["shap_500 vs lime_500"]
holm_survivors = head_to_head[head_to_head["holm_significant"]]
holm_count = int(len(holm_survivors))
holm_shap = int((holm_survivors["first_advantage"] > 0).sum())
holm_lime = int((holm_survivors["first_advantage"] < 0).sum())
comparison_total = int(len(head_to_head))


RANKED_SOURCES = ["shap_500", "lime_500", "occlusion", "shap_50", "lime_50"]
FULL_BUDGET_SOURCES = ["shap_500", "lime_500", "occlusion"]

relative_table = (
    tidy[
        tidy["source"].isin(RANKED_SOURCES)
        & (tidy["combination"] == "absolute_sum")
    ]
    .pivot_table(
        values="relative_shift",
        index=["dataset", "chunk_size"],
        columns=["direction", "source"],
        aggfunc="mean",
    )
    .reindex(
        columns=pd.MultiIndex.from_product(
            [["sufficiency", "comprehensiveness"], RANKED_SOURCES]
        )
    )
    * 100
)
sufficiency_percent_range = (
    float(relative_table["sufficiency"][FULL_BUDGET_SOURCES].min().min()),
    float(relative_table["sufficiency"][FULL_BUDGET_SOURCES].max().max()),
)
comprehensiveness_percent_range = (
    float(relative_table["comprehensiveness"][FULL_BUDGET_SOURCES].min().min()),
    float(relative_table["comprehensiveness"][FULL_BUDGET_SOURCES].max().max()),
)

head_to_head_display = head_to_head.xs("absolute_sum", level="combination").copy()
for column in ["average_shift_first", "average_shift_second", "first_advantage"]:
    head_to_head_display[column] = head_to_head_display[column] * 100

comparison_flat = method_comparison.reset_index()
comparison_flat["significant_for_first"] = (
    comparison_flat["holm_significant"] & (comparison_flat["first_advantage"] > 0)
)
comparison_flat["significant_for_second"] = (
    comparison_flat["holm_significant"] & (comparison_flat["first_advantage"] < 0)
)
MATCHED_COMPARISONS = [
    "shap_500 vs lime_500", "shap_50 vs lime_50",
    "occlusion vs shap_500", "occlusion vs lime_500",
]
matched_view = (
    comparison_flat[comparison_flat["comparison"].isin(MATCHED_COMPARISONS)]
    .groupby(["comparison", "chunk_size", "direction"])
    .agg(
        mean_advantage_percent=(
            "first_advantage", lambda advantages: float(advantages.mean()) * 100
        ),
        significant_for_first=("significant_for_first", "sum"),
        significant_for_second=("significant_for_second", "sum"),
    )
)
BUDGET_COMPARISONS = ["shap_500 vs shap_50", "lime_500 vs lime_50"]
budget_view = (
    comparison_flat[comparison_flat["comparison"].isin(BUDGET_COMPARISONS)]
    .groupby(["comparison", "direction"])
    .agg(
        mean_advantage_percent=(
            "first_advantage", lambda advantages: float(advantages.mean()) * 100
        ),
        significant_for_first=("significant_for_first", "sum"),
        significant_for_second=("significant_for_second", "sum"),
    )
)


def example_shifts(direction):
    subset = tidy[
        (tidy["dataset"] == "imdb")
        & (tidy["source"] == "shap_500")
        & (tidy["chunk_size"] == EXAMPLE_CHUNK_SIZE)
        & (tidy["combination"] == "absolute_sum")
        & (tidy["direction"] == direction)
        & (tidy["observation_index"] == chosen_index)
    ].sort_values("percentile")
    return subset["probability_shift"].tolist()


def shift_list_text(shifts):
    return ", ".join(f"{value:.3f}" for value in shifts)


example_sufficiency_shifts = example_shifts("sufficiency")
example_comprehensiveness_shifts = example_shifts("comprehensiveness")
example_sufficiency_mean = float(np.mean(example_sufficiency_shifts))
example_comprehensiveness_mean = float(np.mean(example_comprehensiveness_shifts))
example_sufficiency_percent = example_sufficiency_mean / uncut_probability * 100
example_comprehensiveness_percent = (
    example_comprehensiveness_mean / uncut_probability * 100
)


def total_minutes(method, budget):
    subset = timing[(timing["method"] == method) & (timing["budget"] == budget)]
    return float(subset["wall_seconds"].sum()) / 60


shap_total_minutes = total_minutes("shap", "500")
lime_total_minutes = total_minutes("lime", "500")
occlusion_total_minutes = total_minutes("occlusion", "word_count")
timing_ratio = lime_total_minutes / shap_total_minutes

total_reviews = OBSERVATION_COUNT * len(DATASET_DIRECTORIES)
accuracy_range = (
    min(metadata["test_accuracy"] for metadata in metadata_by_dataset.values()),
    max(metadata["test_accuracy"] for metadata in metadata_by_dataset.values()),
)


def verdict_sentence(direction):
    counts = verdict_counts[direction]
    return (
        f"{direction} favors SHAP in {counts['shap']} of {counts['total']} conditions "
        f"and LIME in {counts['lime']}, with {counts['noise']} inside sampling noise"
    )


for direction in ["sufficiency", "comprehensiveness"]:
    print(verdict_sentence(direction))
print(f"weakest method versus control advantage: {weakest_control_advantage:+.1f}")
print(f"timing at 500: SHAP {shap_total_minutes:.0f} min, LIME {lime_total_minutes:.0f} min, "
      f"ratio {timing_ratio:.2f}; occlusion {occlusion_total_minutes:.0f} min")
print()
print(comparison_summary.to_string())

sufficiency favors SHAP in 6 of 27 conditions and LIME in 3, with 18 inside sampling noise
comprehensiveness favors SHAP in 4 of 27 conditions and LIME in 0, with 23 inside sampling noise
weakest method versus control advantage: +1.2
timing at 500: SHAP 48 min, LIME 98 min, ratio 2.05; occlusion 27 min

                       conditions  significant_after_holm  favoring_first  favoring_second
comparison                                                                                
shap_500 vs lime_500           54                       8               5                3
shap_50 vs lime_50             54                       7               7                0
occlusion vs shap_500          54                      17               0               17
occlusion vs lime_500          54                      11               0               11
shap_500 vs shap_50            54                       9               9                0
lime_500 vs lime_50            54                      18 

## Assemble and write the report

A single self contained HTML file: styles inline, figures embedded as base64
PNGs, tables rendered from the pandas frames. It opens identically from the
repository, an email attachment, or a static host.

In [10]:
def cite(*reference_numbers):
    return (
        "&#8239;["
        + ", ".join(
            f'<a href="#ref-{number}">{number}</a>' for number in reference_numbers
        )
        + "]"
    )


REFERENCES = [
    'S. M. Lundberg and S.-I. Lee. A unified approach to interpreting model predictions. <i>Advances in Neural Information Processing Systems 30 (NeurIPS)</i>, 2017. <a href="https://arxiv.org/abs/1705.07874">arXiv:1705.07874</a>',
    'M. T. Ribeiro, S. Singh, and C. Guestrin. "Why should I trust you?": Explaining the predictions of any classifier. <i>Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining</i>, pages 1135-1144, 2016. <a href="https://arxiv.org/abs/1602.04938">arXiv:1602.04938</a>',
    'M. D. Zeiler and R. Fergus. Visualizing and understanding convolutional networks. <i>European Conference on Computer Vision (ECCV)</i>, 2014. <a href="https://arxiv.org/abs/1311.2901">arXiv:1311.2901</a>',
    'F. Doshi-Velez and B. Kim. Towards a rigorous science of interpretable machine learning. 2017. <a href="https://arxiv.org/abs/1702.08608">arXiv:1702.08608</a>',
    'C.-K. Yeh, C.-Y. Hsieh, A. S. Suggala, D. I. Inouye, and P. Ravikumar. On the (in)fidelity and sensitivity of explanations. <i>Advances in Neural Information Processing Systems 32 (NeurIPS)</i>, 2019. <a href="https://arxiv.org/abs/1901.09392">arXiv:1901.09392</a>',
    'I. Lage, E. Chen, J. He, M. Narayanan, B. Kim, S. J. Gershman, and F. Doshi-Velez. An evaluation of the human-interpretability of explanation. 2019. <a href="https://arxiv.org/abs/1902.00006">arXiv:1902.00006</a>',
    'D. Nguyen. Comparing automatic and human evaluation of local explanations for text classification. <i>Proceedings of NAACL-HLT</i>, pages 1069-1078, 2018. <a href="https://aclanthology.org/N18-1097/">ACL Anthology N18-1097</a>',
    'J. DeYoung, S. Jain, N. F. Rajani, E. Lehman, C. Xiong, R. Socher, and B. C. Wallace. ERASER: A benchmark to evaluate rationalized NLP models. <i>Proceedings of ACL</i>, pages 4443-4458, 2020. <a href="https://aclanthology.org/2020.acl-main.408/">ACL Anthology 2020.acl-main.408</a>',
    'A. Jacovi and Y. Goldberg. Towards faithfully interpretable NLP systems: How should we define and evaluate faithfulness? <i>Proceedings of ACL</i>, pages 4198-4205, 2020. <a href="https://aclanthology.org/2020.acl-main.386/">ACL Anthology 2020.acl-main.386</a>',
    'P. Atanasova, J. G. Simonsen, C. Lioma, and I. Augenstein. A diagnostic study of explainability techniques for text classification. <i>Proceedings of EMNLP</i>, pages 3256-3274, 2020. <a href="https://aclanthology.org/2020.emnlp-main.263/">ACL Anthology 2020.emnlp-main.263</a>',
    'D. Alvarez-Melis and T. S. Jaakkola. On the robustness of interpretability methods. <i>ICML Workshop on Human Interpretability in Machine Learning (WHI)</i>, 2018. <a href="https://arxiv.org/abs/1806.08049">arXiv:1806.08049</a>',
    'M. Naylor, C. French, S. Terker, and U. Kamath. Quantifying explainability in NLP and analyzing algorithms for performance-explainability tradeoff. <i>ICML Workshop on Interpretable Machine Learning in Healthcare</i>, 2021. <a href="https://arxiv.org/abs/2107.05693">arXiv:2107.05693</a>',
    'U. Bhatt, A. Weller, and J. M. F. Moura. Evaluating and aggregating feature-based model explanations. <i>Proceedings of IJCAI</i>, 2020. <a href="https://arxiv.org/abs/2005.00631">arXiv:2005.00631</a>',
    'S. Hooker, D. Erhan, P.-J. Kindermans, and B. Kim. A benchmark for interpretability methods in deep neural networks. <i>Advances in Neural Information Processing Systems 32 (NeurIPS)</i>, pages 9737-9748, 2019. <a href="https://arxiv.org/abs/1806.10758">arXiv:1806.10758</a>',
    'J. Adebayo, J. Gilmer, M. Muelly, I. Goodfellow, M. Hardt, and B. Kim. Sanity checks for saliency maps. <i>Advances in Neural Information Processing Systems 31 (NeurIPS)</i>, 2018. <a href="https://arxiv.org/abs/1810.03292">arXiv:1810.03292</a>',
    'M. Sundararajan, A. Taly, and Q. Yan. Axiomatic attribution for deep networks. <i>Proceedings of ICML</i>, 2017. <a href="https://arxiv.org/abs/1703.01365">arXiv:1703.01365</a>',
    'S. Jain and B. C. Wallace. Attention is not explanation. <i>Proceedings of NAACL-HLT</i>, pages 3543-3556, 2019. <a href="https://aclanthology.org/N19-1357/">ACL Anthology N19-1357</a>',
    'S. Wiegreffe and Y. Pinter. Attention is not not explanation. <i>Proceedings of EMNLP-IJCNLP</i>, pages 11-20, 2019. <a href="https://aclanthology.org/D19-1002/">ACL Anthology D19-1002</a>',
    'J. Bastings and K. Filippova. The elephant in the interpretability room: Why use attention as explanation when we have saliency methods? <i>Proceedings of the Third BlackboxNLP Workshop</i>, pages 149-155, 2020. <a href="https://aclanthology.org/2020.blackboxnlp-1.14/">ACL Anthology 2020.blackboxnlp-1.14</a>',
    'W. Samek, A. Binder, G. Montavon, S. Lapuschkin, and K.-R. M&uuml;ller. Evaluating the visualization of what a deep neural network has learned. <i>IEEE Transactions on Neural Networks and Learning Systems</i>, 28(11):2660-2673, 2017. <a href="https://arxiv.org/abs/1509.06321">arXiv:1509.06321</a>',
    'R. Dror, G. Baumer, S. Shlomov, and R. Reichart. The hitchhiker\'s guide to testing statistical significance in natural language processing. <i>Proceedings of ACL</i>, pages 1383-1392, 2018. <a href="https://aclanthology.org/P18-1128/">ACL Anthology P18-1128</a>',
    'J. Zhou, A. H. Gandomi, F. Chen, and A. Holzinger. Evaluating the quality of machine learning explanations: A survey on methods and metrics. <i>Electronics</i>, 10(5):593, 2021. <a href="https://www.mdpi.com/2079-9292/10/5/593">MDPI 10.3390/electronics10050593</a>',
]

REPORT_CSS = '''
:root { color-scheme: light; }
body {
  background: #f9f9f7; color: #0b0b0b; margin: 0;
  font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
  line-height: 1.55;
}
main { max-width: 60rem; margin: 0 auto; padding: 2.5rem 1.5rem 4rem; }
h1 { font-size: 1.9rem; line-height: 1.2; margin: 0 0 0.4rem; }
h2 { font-size: 1.35rem; margin: 2.6rem 0 0.8rem; }
h3 { font-size: 1.05rem; margin: 1.8rem 0 0.5rem; }
p, li { max-width: 46rem; }
.byline { color: #52514e; margin-bottom: 2rem; }
.abstract {
  background: #fcfcfb; border: 1px solid rgba(11,11,11,0.10);
  border-radius: 8px; padding: 1.1rem 1.4rem; margin: 1.5rem 0;
}
.abstract p { margin: 0.4rem 0; }
figure { margin: 1.6rem 0; }
figure img { max-width: 100%; height: auto; background: #fcfcfb;
  border: 1px solid rgba(11,11,11,0.10); border-radius: 8px; }
figcaption { color: #52514e; font-size: 0.9rem; margin-top: 0.5rem; max-width: 46rem; }
.table-wrap { overflow-x: auto; margin: 1.2rem 0; }
table { border-collapse: collapse; font-size: 0.85rem; background: #fcfcfb; }
th, td { padding: 0.35rem 0.7rem; border: 1px solid #e1e0d9; text-align: right;
  font-variant-numeric: tabular-nums; }
th { background: #f0efec; color: #0b0b0b; text-align: right; }
td:first-child, th:first-child { text-align: left; }
code { background: #f0efec; padding: 0.1rem 0.3rem; border-radius: 4px;
  font-size: 0.85em; }
a { color: #256abf; }
.formula { font-family: Georgia, "Times New Roman", serif; font-style: normal;
  text-align: center; margin: 1.1rem auto; font-size: 1.02rem; }
.references { font-size: 0.88rem; padding-left: 1.4rem; }
.references li { margin-bottom: 0.5rem; max-width: 46rem; }
.footer { color: #898781; font-size: 0.85rem; margin-top: 3rem;
  border-top: 1px solid #e1e0d9; padding-top: 1rem; }
'''


def frame_to_html(frame, caption):
    table_html = frame.to_html(
        float_format=lambda value: f"{value:.3f}", border=0
    )
    return (
        f'<div class="table-wrap">{table_html}</div>'
        f'<figcaption>{caption}</figcaption>'
    )


setup_rows = []
for dataset_name, metadata in metadata_by_dataset.items():
    dataset_tidy = tidy[tidy["dataset"] == dataset_name]
    setup_rows.append(
        {
            "dataset": dataset_name,
            "task": DATASET_TASKS[dataset_name],
            "train / test": f"{metadata['train_size']} / {metadata['test_size']}",
            "test accuracy": metadata["test_accuracy"],
            "explained reviews": OBSERVATION_COUNT,
        }
    )
setup_table = pd.DataFrame(setup_rows).set_index("dataset")

timing_display = timing.set_index(["dataset", "method", "budget"])[
    ["wall_seconds", "seconds_per_review", "prediction_count", "predictions_per_second"]
]

sections = []

sections.append(f'''
<h1>Quantifying the Faithfulness of Text Classification Explanations</h1>
<p class="byline">An equal budget benchmark of SHAP, LIME and occlusion on three datasets &middot; generated from the shapley pipeline, notebooks 01 to 07 &middot; seed {RANDOM_SEED} &middot; {OBSERVATION_COUNT} reviews per dataset</p>

<div class="abstract">
<h3 style="margin-top:0">Abstract</h3>
<p>Feature attribution methods promise to show which words drive a text
classifier's prediction, but different methods produce visibly different
explanations, so at most one of them can be taken at face value. We compare
SHAP's partition explainer, LIME, and the leave one word out occlusion
baseline on three binary classification tasks: movie review sentiment (IMDB),
business review sentiment (Yelp polarity) and news topic classification
(AG News, business vs sci/tech), each served by an identical frozen encoder
pipeline with test accuracy between {accuracy_range[0]:.2f} and
{accuracy_range[1]:.2f}. SHAP and LIME run at an equal budget of
{EVALUATION_BUDGET} model evaluations per explained text and again at a tenth
of it; occlusion's cost is structural, one prediction per word.
Faithfulness is measured by deletion: words are removed in chunks ranked by
attributed importance, and the model's probability shift is recorded, in the
sufficiency direction (removing unimportant text should not move the
prediction) and the comprehensiveness direction (removing important text
should move it), at removal thresholds of 20 to 80 percent, against each
method's own shuffled control.</p>
<p>At the full budget, both methods outrank their randomized controls in
every condition: all {control_total} paired control contrasts are positive,
{control_significant} of them beyond the two standard error noise threshold,
the weakest at {weakest_control_advantage:+.1f}. Head to head the picture is
task dependent: {verdict_sentence("sufficiency")};
{verdict_sentence("comprehensiveness")}. The significant differences
concentrate at chunk size 1, where word level sufficiency favors SHAP on both
sentiment tasks but LIME on the news topic task. Occlusion never beats a
budgeted method: it loses
{comparison_summary.loc["occlusion vs shap_500", "favoring_second"]} of 54
Holm corrected conditions to SHAP and
{comparison_summary.loc["occlusion vs lime_500", "favoring_second"]} to LIME
and wins none, so its blindness to word interactions is a measured deficit,
not a theoretical one. The tenfold budget cut is survivable for SHAP, which
loses ground in
{comparison_summary.loc["shap_500 vs shap_50", "favoring_first"]} conditions,
and costly for LIME, which loses in
{comparison_summary.loc["lime_500 vs lime_50", "favoring_first"]}; at 50
evaluations the head to head flips to SHAP outright,
{comparison_summary.loc["shap_50 vs lime_50", "favoring_first"]} conditions
to {comparison_summary.loc["shap_50 vs lime_50", "favoring_second"]}. Despite
near identical prediction counts, explaining all {total_reviews} reviews took
{shap_total_minutes:.0f} minutes with SHAP and {lime_total_minutes:.0f}
minutes with LIME on the same 8 core CPU, a ratio of {timing_ratio:.2f};
occlusion took {occlusion_total_minutes:.0f}.</p>
</div>
''')

sections.append(f'''
<h2>1. Experimental setup</h2>
<p>Three binary text classification datasets, one shared pipeline. Each
dataset is reduced to a stratified sample of 2000 documents, split 80/20,
embedded with the frozen <code>intfloat/e5-small-v2</code> encoder (mean
pooled, L2 normalized, 512 token limit), and classified by a single hidden
layer MLP head trained on the embeddings. The classifier is deliberately
simple: it exists to be explained, and keeping the pipeline constant across
datasets means differences in the measurements are attributable to the data
and the attribution methods rather than to modeling choices.</p>
{frame_to_html(setup_table, "Table 1. Datasets and models. Every pipeline choice, including the random seed, is shared across datasets.")}
<p>From each test split, {OBSERVATION_COUNT} documents are sampled with label
stratification (seed {RANDOM_SEED}) and explained by both methods:</p>
<ul>
<li><b>SHAP</b>{cite(1)}, the partition explainer over a whitespace word
masker, capped at {EVALUATION_BUDGET} model evaluations per document. The cap
is a budget, not an exact count: on short documents the partition tree
resolves completely and stops early, which the measured prediction counts in
Table 2 make visible.</li>
<li><b>LIME</b>{cite(2)}, ridge regression over {EVALUATION_BUDGET} random
word dropout samples per document, positional features
(<code>bow=False</code>), dropped words replaced by the same
<code>...</code> marker the masker uses.</li>
<li><b>Occlusion</b>{cite(3)}, the leave one word out baseline: the value of a word is
the drop in predicted class probability when that word alone is masked. It
has no budget knob, its cost is fixed at one prediction per word plus one for
the original, which places it between the two budgets on these datasets and
makes it the natural fixed cost reference.</li>
</ul>
<p>To measure how explanation power scales with budget, SHAP and LIME also
run at 50 evaluations per document, a tenth of the main budget. All methods
explain the predicted class and produce one signed value per whitespace word,
so every downstream measurement treats them identically. Timings were
recorded on the same otherwise loaded 8 core CPU machine (Intel Core Ultra 7
258V, CPU only PyTorch), so they are indicative rather than laboratory
grade.</p>
''')

sections.append(f'''
<h2>2. Related work: how explanations are evaluated, and why most metrics resist text</h2>
<p>Doshi-Velez and Kim{cite(4)} sort interpretability evaluation into
application grounded (domain experts on real tasks), human grounded (lay
subjects on simplified tasks) and functionally grounded (formal proxies, no
humans). Yeh et al.{cite(5)} draw the same line for feature attributions
between subjective and objective measures, and observe that subjective
measures have been the predominant practice. This benchmark sits deliberately
on the functionally grounded, objective side, because its question, which
attribution method ranks words more faithfully and at what cost, is answered
by tens of thousands of scored deletions per method that no human protocol
could price. The families below explain what that choice gives up and what it
escapes.</p>

<p><b>Human subject metrics.</b> The human grounded family measures what
people can do with an explanation: simulate the model's output, verify a
claim, reason about a counterfactual. Lage et al.{cite(6)} is the careful
version of that program, and its costs are instructive: nine hundred
recruited participants, two synthetic domains built so participants could
bring no prior knowledge, response time, accuracy and satisfaction as
outcomes, and, after all of it, no statistically significant effect of
explanation variation on accuracy, their correctness signal. For text
classification specifically, Nguyen{cite(7)} compared crowdsourced judgments
with automatic word deletion measures and found only moderate correlation,
with human judgments swayed by factors beyond the explanation itself. Human
evaluation answers a question this paper does not ask, whether explanations
help people, and it cannot be rerun overnight when the model, dataset or
method changes.</p>

<p><b>Plausibility against human rationales.</b> ERASER{cite(8)} scores text
explanations by agreement with human annotated rationales, token F1 and
intersection over union. This requires a rationale annotated corpus, which
exists for a handful of datasets and must be built anew for each addition,
and it measures plausibility rather than faithfulness: Jacovi and
Goldberg{cite(9)} argue the two must not be conflated, since a model may
genuinely rely on evidence no human would mark. Atanasova et al.{cite(10)},
the closest prior diagnostic study for text classification, mix agreement
with human rationales into their battery alongside faithfulness properties.
Our benchmark added two datasets in one notebook without annotating a single
rationale, which is the practical difference.</p>

<p><b>Infidelity, sensitivity and robustness.</b> A third family is defined
over continuous input geometry. Infidelity{cite(5)} scores an attribution by
how well inner products with random perturbations predict the model's output
change; robustness{cite(11)} asks that near identical inputs receive near
identical attributions, formalized as local Lipschitz continuity inside an
epsilon ball. Both are objective and automatic where they apply, and both
presuppose operations raw text does not have: a perturbation that can be
arithmetically subtracted from the input, and a neighborhood radius that
means the same thing across representations. Naylor et al.{cite(12)}, whose
title this paper echoes, ported both metrics to clinical text through
embedding space token replacement and documented where they crack: no radius
transfers across text representations, and on inherently interpretable
models the ground truth explanation sometimes scored worse on infidelity
than post hoc surrogates, which the authors read as the metric mis
transferring rather than as a fact about the explanations. Bhatt et
al.{cite(13)}, whose catalogue of quantitative desiderata is the broadest,
formulate faithfulness as correlation over feature subsets set to a
reference baseline, again a vector text lacks. Alvarez-Melis and Jaakkola's
own escape hatch for discrete inputs, a neighborhood over observed dataset
points{cite(11)}, concedes the same point.</p>

<p><b>Retraining, sanity checks and axioms.</b> Remove and retrain,
ROAR{cite(14)}, deletes the top ranked features from the whole corpus and
retrains the model, which controls for the distribution shift that deletion
introduces but costs one full training run per method, threshold and seed,
and measures a property of the dataset and training procedure as much as of
the explanation. Sanity checks{cite(15)} randomize model weights and require
attributions to change, and axiomatic analyses{cite(16)} certify properties
such as completeness and implementation invariance; both are necessary
conditions that screen out broken methods but do not rank working ones.
Attention weights, the one free explanation transformers offer, are contested
as explanation at all{cite(17, 18)}; Bastings and Filippova{cite(19)} argue
that when the question is which input tokens mattered, saliency methods with
explicit faithfulness semantics are the right tool, and this benchmark
accordingly evaluates only such methods.</p>

<p><b>Where this work stands.</b> The deletion family we build on descends
from region perturbation curves for images{cite(20)} and reaches text
classification as ERASER's sufficiency and comprehensiveness{cite(8)}. Its
attraction is exactly what the families above lack in the text setting:
deleting whole words through a masker needs no baseline vector, no epsilon
ball, no arithmetic on tokens and no human annotation, and it can be
recomputed for any new dataset, model or method at the price of model
predictions alone. Its known weakness is that deleted text is out of
distribution{cite(14)}, so some output movement is guaranteed regardless of
explanation quality. This benchmark's response is threefold. Every
attribution source is paired with its own shuffled control, which suffers the
identical distribution shift and so isolates ranking quality, the graded,
comparative evaluation Jacovi and Goldberg call for{cite(9)}. Every method is
scored under the same masker its attributions were computed with, at an
equal, counted evaluation budget, so no method is judged against perturbation
semantics it never saw or with compute another method did not get. And the
per review pairing carries into the statistics, one paired test per condition
on threshold averaged scores with family wise Holm correction, following
standard significance testing guidance for NLP{cite(21)} and the multiple
comparisons discipline of the human subject literature{cite(6)}. Zhou et
al.'s survey{cite(22)} concludes that no single metric covers all aspects of
interpretability, and this work does not claim otherwise: it deliberately
measures neither plausibility nor human usability. The claim is narrower and
checkable, that deletion faithfulness under matched budgets and matched
controls is the fragment of explanation quality that is fully automatic for
text classification, and that it suffices to answer which method ranks words
more faithfully, and at what computational cost.</p>
''')

sections.append(f'''
<h2>3. Faithfulness: definition</h2>
<p>An attribution is faithful when the text it calls important is the text
the model actually relies on. The measurement deletes spans and watches the
output. Words are grouped into chunks of 1, 4 or 8 consecutive words, each
chunk is scored from its word attributions, chunks are ranked, and the lowest
or highest ranked chunks are removed by the same masker that produced the
attributions, with removed spans collapsing to a <code>...</code> marker. A
threshold of 40 means the 40th percentile of that document's own chunk
scores, so roughly 40 percent of its text is removed.</p>
<ul>
<li><b>Sufficiency</b>{cite(8)}: remove the <i>least</i> important chunks. A faithful
ranking discards only noise, so the absolute probability shift should be
small. Lower is better.</li>
<li><b>Comprehensiveness</b>{cite(8)}: remove the <i>most</i> important chunks. A
faithful ranking identifies what the model depends on, so the shift should be
large. Higher is better.</li>
</ul>
<p>Chunk scores combine word attributions in three ways, which disagree
exactly when a chunk holds values that cancel: <code>absolute_sum</code>, the
magnitude of the summed signed values, the theoretically motivated choice for
additive attributions; <code>sum_absolute</code>, total magnitude regardless
of direction; and <code>max_absolute</code>, the single strongest word.</p>
<h3>3.1 Formal definition</h3>
<p>Write <i>f</i>(<i>s</i>) for the model's probability of the predicted
class on text <i>s</i>, the class fixed once from the full text. Fix a chunk
size, a combination, and cut thresholds
<i>t</i><sub>1</sub> &lt; &hellip; &lt; <i>t</i><sub><i>n</i></sub>, here 20,
40, 60 and 80. For a threshold <i>t</i>, let
<i>s</i>&#8725;<i>t</i><sup>&minus;</sup> denote <i>s</i> with every chunk
whose importance falls below the <i>t</i>-th percentile of <i>s</i>'s own
chunk scores removed, and <i>s</i>&#8725;<i>t</i><sup>+</sup> denote
<i>s</i> with every chunk at or above the (100&minus;<i>t</i>)-th percentile
removed. The two scores average the model's movement over the thresholds:</p>
<p class="formula">
sufficiency(<i>s</i>) = (1/<i>n</i>) &Sigma;<sub><i>i</i>=1&hellip;<i>n</i></sub>
&#124;<i>f</i>(<i>s</i>) &minus; <i>f</i>(<i>s</i>&#8725;<i>t</i><sub><i>i</i></sub><sup>&minus;</sup>)&#124;
</p>
<p class="formula">
comprehensiveness(<i>s</i>) = (1/<i>n</i>) &Sigma;<sub><i>i</i>=1&hellip;<i>n</i></sub>
&#124;<i>f</i>(<i>s</i>) &minus; <i>f</i>(<i>s</i>&#8725;<i>t</i><sub><i>i</i></sub><sup>+</sup>)&#124;
</p>
<p>The relative forms divide by the model's original confidence,
sufficiency<sub>rel</sub>(<i>s</i>) = sufficiency(<i>s</i>) / <i>f</i>(<i>s</i>)
and likewise for comprehensiveness, so a score reads as the share of the
model's confidence the cuts moved; because the predicted class starts at or
above one half, both relative scores lie in [0, 1]. Sufficiency is better
when lower, comprehensiveness when higher.</p>
<p><b>A numerical example of both.</b> Take the review of Section 5,
<i>f</i>(<i>s</i>) = {uncut_probability:.3f}, explained by SHAP at the full
budget, chunk size {EXAMPLE_CHUNK_SIZE}, <code>absolute_sum</code>. The four
sufficiency cuts moved the model by
&#124;<i>f</i>(<i>s</i>) &minus; <i>f</i>(<i>s</i>&#8725;<i>t</i><sup>&minus;</sup>)&#124; =
{shift_list_text(example_sufficiency_shifts)} at <i>t</i> = 20, 40, 60, 80;
the mean is {example_sufficiency_mean:.3f}, and dividing by
<i>f</i>(<i>s</i>) gives a relative sufficiency of
{example_sufficiency_percent:.1f} percent. The four comprehensiveness cuts
moved it by {shift_list_text(example_comprehensiveness_shifts)}; the mean is
{example_comprehensiveness_mean:.3f}, a relative comprehensiveness of
{example_comprehensiveness_percent:.1f} percent. The asymmetry between the
two numbers is the faithfulness signal for this review: the words this
explanation dismisses barely move the model, the words it crowns move it
substantially.</p>

<p>Neither direction means anything in isolation, because deleted text pushes
the input off the training distribution{cite(14)} and some shift is guaranteed. Every
condition is therefore paired with a control in which that method's own
attribution values are randomly permuted within each document, preserving the
value distribution while destroying the ranking. All comparisons are paired
per document: the reported advantage is the mean paired difference divided by
its standard error, averaged over thresholds, so magnitudes above roughly 2
exceed sampling noise.</p>
''')

sections.append(f'''
<h2>4. Results</h2>

<h3>4.1 Speed of execution</h3>
<figure>
<img src="{report_images["timing"]}" alt="Bar chart of attribution cost in seconds per review, per dataset and method">
<figcaption>Figure 1. Wall clock cost per explained review. Both methods pay
almost entirely in model predictions; the prediction counts in Table 2 show
the budget parity held almost exactly.</figcaption>
</figure>
{frame_to_html(timing_display.round(2), "Table 2. Timing and measured prediction counts for computing " + str(OBSERVATION_COUNT) + " attributions per dataset.")}
<p>The budget parity is visible in the prediction counts: at each budget,
SHAP and LIME made within one percent of the same number of predictions, yet
LIME took roughly twice as long; occlusion's count is simply each dataset's
word count plus one original per review. Two mechanisms account for the
timing gap. SHAP's masker
collapses each removed span into a single <code>...</code> marker, so its
perturbed texts shrink as the partition tree descends, while LIME keeps one
marker per dropped word and its perturbed texts stay near full length for the
encoder. LIME additionally pays for a ridge fit and a distance computation
per document. The gap narrows on AG News, whose texts are short enough that
per call overhead rather than sequence length dominates.</p>

<h3>4.2 Faithfulness curves</h3>
<figure>
<img src="{report_images["curves"]}" alt="Line charts of sufficiency and comprehensiveness versus percent of text cut, for each dataset">
<figcaption>Figure 2. Mean shift as a percent of the model's confidence as the cut deepens,
shown for <code>absolute_sum</code> at chunk size 4; the full grid is in
Table 3. Solid lines are the methods, dashed lines their shuffled controls.
In the top row lower is better and the gap below the dashed control is the
faithfulness signal; in the bottom row higher is better.</figcaption>
</figure>

<h3>4.3 The full grid</h3>
{frame_to_html(paired_grid.round(2), "Table 3. Mean shifts, head to head paired advantages (positive favors SHAP), and per method advantages over shuffled controls, for every dataset, chunk size and combination.")}
<figure>
<img src="{report_images["advantage"]}" alt="Matrix of paired advantages per dataset and condition, colored by which method they favor">
<figcaption>Figure 3. Head to head paired advantage for every condition.
Blue favors SHAP, orange favors LIME, bold values exceed the noise threshold
of two standard errors.</figcaption>
</figure>
<p>Two patterns stand out. First, the decisive cells sit almost entirely at
chunk size 1: aggregating words into chunks of 4 or 8 washes the methods'
differences into noise, consistent with the intuition that chunking averages
away per word estimation error. Second, the word level winner flips with the
task: SHAP's ranking is harder to beat on the two sentiment tasks, LIME's on
the news topic task, where the signal concentrates in a few entity like terms
that LIME's sparse linear surrogate is well shaped to isolate. Neither method
dominates the grid.</p>

<h3>4.4 One score per review: averaging over the thresholds</h3>
<p>Averaging each review's shift over the four thresholds collapses the
deletion curve into a single score per review, the area over the perturbation
curve{cite(20, 8)}, here on the relative scale of Section 3.1 so every number
reads as percent of the model's confidence. This makes the head to head test cleaner:
one paired t statistic and one Wilcoxon signed rank test per condition, on
{OBSERVATION_COUNT} paired scores, with a Holm correction across all
{comparison_total} conditions. After correction, {holm_count} conditions
separate the methods: {holm_lime} favor LIME, all of them word level
sufficiency on the news topic task, and {holm_shap} favor SHAP, all of them
chunked comprehensiveness on Yelp. The mean level sufficiency advantages SHAP
shows at the word level on the sentiment tasks (Figure 3) do not survive the
rank based test, a sign they are carried by a minority of reviews with large
differences rather than by a consistent per review edge.</p>
{frame_to_html(head_to_head_display.round(2), "Table 4. Threshold averaged SHAP versus LIME at budget 500, absolute_sum slice, on the relative scale: shifts and advantages are percent of the model's original confidence, positive first_advantage favors SHAP. The full table over all combinations and all six pairwise comparisons is saved as method_comparison.parquet next to the tidy results.")}

<p>Raw shifts are probability points and inherit each model head's
calibration, the Yelp head is far more confident than IMDB's, so Table 5
restates the central results as a percentage of each review's original
predicted class probability, the share of the model's confidence the cut
moved. Read this way the two directions separate starkly: at the full budget
and under <code>absolute_sum</code>, cutting the least important text moves
predictions by {sufficiency_percent_range[0]:.0f} to
{sufficiency_percent_range[1]:.0f} percent of the model's confidence on
average, while cutting the most important text moves them by
{comprehensiveness_percent_range[0]:.0f} to
{comprehensiveness_percent_range[1]:.0f} percent. The tidy results carry the
per observation <code>relative_shift</code> column, so any statistic in this
report can be re-derived on the relative scale; the paired per review signs
are identical on both scales because every source shares an observation's
denominator.</p>
{frame_to_html(relative_table.round(1), "Table 5. Mean shift as a percent of the model's original predicted class probability, absolute_sum, averaged over thresholds and observations, for every attribution source.")}
<p>Splitting the head to head comparisons by what is held fixed makes the
structure legible. Table 6 holds the evaluation budget and the chunk size
fixed and asks which method wins, aggregating each row over the three
datasets and three combinations: the method question, answered per
granularity. Chunking visibly dilutes every method contrast, and occlusion's
deficit is largest exactly where explanations are usually read, at the word
level.</p>
{frame_to_html(matched_view.round(2), "Table 6. Methods against methods with budget and chunk size held fixed. Mean advantage in percentage points of the model's confidence, positive favoring the first named source; significant counts are Holm corrected within each comparison family, nine conditions per row.")}

<h3>4.5 How different are the explanations?</h3>
{frame_to_html(rank_correlations.round(2), "Table 7. Median per document Spearman rank correlation between the attribution methods at budget 500, occlusion at its fixed cost.")}
<p>Agreement between the methods tracks how concentrated the task's signal
is: the median SHAP to LIME rank correlation rises from
{rank_correlations.loc["imdb", "shap_vs_lime"]:.2f} on IMDB, where sentiment
spreads across many words, to
{rank_correlations.loc["agnews", "shap_vs_lime"]:.2f} on AG News, where a few
entity terms decide the class. Even at the high end the rankings are far from
identical, which makes the comparison above substantive rather than
tautological: these are different explanations, not reparameterizations of
one another.</p>

<h3>4.6 Power versus the evaluation budget</h3>
<p>How many model evaluations does a faithful explanation need? Each method
appears at its two budgets, occlusion at its fixed per word cost, plotted
against the predictions each source actually made. Chunk size 1 under
<code>absolute_sum</code> is where the methods differ most and where all
three combinations coincide, so it is the honest place to read budget
effects. LIME at 50 samples fits its ridge surrogate on fewer samples than
most documents have words, while SHAP at 50 evaluations barely splits its
partition tree, so the cheap arms probe two different failure modes.</p>
<figure>
<img src="{report_images["budget"]}" alt="Faithfulness scores versus predictions per review, with each method at two budgets and occlusion at its fixed cost">
<figcaption>Figure 4. Threshold averaged faithfulness, in percent of the model's
confidence, as a function of the evaluations actually spent, chunk size 1,
<code>absolute_sum</code>. Points
left to right per method: budget 50, budget 500; the diamond is occlusion at
its natural per word cost.</figcaption>
</figure>
{frame_to_html(comparison_summary, "Table 8. Holm corrected outcome of every pairwise comparison over all 54 conditions each: how many conditions separate the pair significantly, and in whose favor.")}
<p>Collapsing instead over everything except the direction isolates the
budget question per method, both metrics side by side:</p>
{frame_to_html(budget_view.round(2), "Table 9. Each method against its own reduced budget, twenty seven conditions per row. Mean advantage in percentage points of confidence, positive favoring the full budget.")}
<p>The outcome is one sided in two ways. Occlusion, despite spending more
predictions than the 50 budget arms on the longer datasets, never wins a
corrected condition against either full budget method: judging each word in
isolation misses redundancy and interaction, and the deletion tests price
that blindness directly. And the budgets are not symmetric between methods.
LIME loses
{comparison_summary.loc["lime_500 vs lime_50", "favoring_first"]} of 54
conditions to its own full budget arm, twice SHAP's
{comparison_summary.loc["shap_500 vs shap_50", "favoring_first"]}, because a
ridge surrogate fitted on 50 dropout samples is underdetermined for documents
with more words than samples, while SHAP's partition estimates merely get
coarser. At the cheap budget the head to head accordingly flips to SHAP
outright,
{comparison_summary.loc["shap_50 vs lime_50", "favoring_first"]} conditions
to {comparison_summary.loc["shap_50 vs lime_50", "favoring_second"]}.</p>
''')

sections.append(f'''
<h2>5. Worked example: the same review under all three methods</h2>
<p>One IMDB review, predicted {class_name} with probability
{uncut_probability:.3f}, explained by all three methods from the benchmark
caches. The SHAP to LIME Spearman rank correlation on this review is
{example_rank_correlation:.2f}.</p>
<figure>
<img src="{report_images["example_maps"]}" alt="The same review rendered three times with per word attribution colors, for SHAP, LIME and occlusion">
<figcaption>Figure 5. Word importance for the same review. Blue pushes toward
the predicted class, orange against it, intensity is magnitude relative to the
strongest word within each panel.</figcaption>
</figure>
<figure>
<img src="{report_images["example_cuts"]}" alt="The same review with the words each method cuts at the 40 percent threshold grayed out, with resulting predictions">
<figcaption>Figure 6. The sufficiency cut each method actually makes at the
{EXAMPLE_PERCENTILE} percent threshold, chunk size {EXAMPLE_CHUNK_SIZE},
<code>absolute_sum</code>. Gray words are removed. The predictions in the
panel titles show how far the model moves relative to its uncut
{uncut_probability:.3f}.</figcaption>
</figure>
''')

sections.append(f'''
<h2>6. Conclusion</h2>
<p>Under an equal evaluation budget, both SHAP and LIME rank text more
faithfully than chance in every condition tested: all {control_total} paired
contrasts against shuffled controls are positive and {control_significant}
exceed the noise threshold, with the weakest at
{weakest_control_advantage:+.1f} standard errors. Head to head,
{verdict_sentence("sufficiency")}, and {verdict_sentence("comprehensiveness")}.
The decisive cells concentrate at the word level and flip with the task,
SHAP ahead on both sentiment datasets, LIME ahead on the news topic dataset,
while chunked conditions are almost uniformly within noise. The threshold
averaged scores sharpen this further: after Holm correction only
{holm_count} of {comparison_total} comparisons separate the methods,
{holm_lime} for LIME (word level sufficiency on the topic task) and
{holm_shap} for SHAP (chunked comprehensiveness on Yelp). Neither method
dominates.</p>
<p>The budget arm turns those observations into practical guidance. Cutting
the budget tenfold never helps: the full budget wins
{comparison_summary.loc["shap_500 vs shap_50", "favoring_first"]} of 54 Holm
corrected conditions for SHAP and
{comparison_summary.loc["lime_500 vs lime_50", "favoring_first"]} for LIME,
the cheap arms none, and at 50 evaluations the method comparison flips
decisively to SHAP
({comparison_summary.loc["shap_50 vs lime_50", "favoring_first"]} conditions
to {comparison_summary.loc["shap_50 vs lime_50", "favoring_second"]}).
Occlusion, whose cost is one prediction per word, never wins a corrected
condition against either full budget method, losing
{comparison_summary.loc["occlusion vs shap_500", "favoring_second"]} to SHAP
and {comparison_summary.loc["occlusion vs lime_500", "favoring_second"]} to
LIME (Tables 8 and 9, Figure 4). When evaluations are scarce, these results say to
spend them on SHAP; and occlusion's simplicity buys strictly less
faithfulness than the same predictions spent on either budgeted method.</p>
<p>The practical reading: at matched budgets, average deletion faithfulness
does not separate these methods nearly as much as their folklore suggests,
so the deciding factors are the task, the granularity the explanation will be
read at, and cost. On cost the result is unambiguous: with near identical
prediction counts SHAP ran about twice as fast overall
(ratio {timing_ratio:.2f}), because its collapsed masking keeps perturbed
texts short while LIME's per word markers keep them near full length. Since
the methods rank words differently (Table 7) while all passing the deletion
tests, deletion faithfulness at this observation count does not by itself
identify a single best explanation; stronger separation would need larger
samples, deeper budgets, or finer grained tests.</p>
<p>Limitations: one model family (frozen encoder plus MLP head), 100
documents per dataset, a whitespace word masker with <code>...</code>
replacement whose out of distribution effects are only partially neutralized
by the shuffled controls, and wall clock timings from a shared machine.</p>
''')

reference_items = "\n".join(
    f'<li id="ref-{number}">{entry}</li>'
    for number, entry in enumerate(REFERENCES, start=1)
)
sections.append(f'''
<h2>References</h2>
<ol class="references">
{reference_items}
</ol>
''')

sections.append(f'''
<div class="footer">
<p>Reproduction: notebooks <code>01_train</code> (IMDB pipeline),
<code>05_additional_datasets</code> (Yelp, AG News),
<code>06_benchmark</code> (attributions, sweep, tidy results),
<code>07_report</code> (this document). All caches live under
<code>artifacts/</code>; tidy results in
<code>faithfulness_results.parquet</code>, timing in
<code>timing_results.parquet</code>. Random seed {RANDOM_SEED} throughout.
Attribution budgets: {EVALUATION_BUDGET} and 50 evaluations per document for
SHAP and LIME, occlusion at one prediction per word. Thresholds
{PERCENTILES}, chunk sizes {CHUNK_SIZES}, combinations {COMBINATIONS}.
Pairwise statistics in <code>method_comparison.parquet</code>.</p>
</div>
''')

report_body = "\n".join(sections)

import re

cited_numbers = {int(number) for number in re.findall(r'#ref-(\d+)"', report_body)}
listed_numbers = set(range(1, len(REFERENCES) + 1))
assert cited_numbers == listed_numbers, (
    f"citation mismatch: cited but unlisted {cited_numbers - listed_numbers}, "
    f"listed but never cited {listed_numbers - cited_numbers}"
)

report_document = (
    "<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n"
    "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n"
    "<title>Quantifying the Faithfulness of Text Classification Explanations</title>\n"
    f"<style>{REPORT_CSS}</style>\n</head>\n<body>\n<main>\n"
    f"{report_body}\n</main>\n</body>\n</html>\n"
)

REPORTS_DIRECTORY.mkdir(exist_ok=True)
REPORT_PATH.write_text(report_document)
print(f"Wrote {REPORT_PATH} ({len(report_document) / 1024:.0f} KiB, "
      f"{len(report_images)} embedded figures, {len(REFERENCES)} references, "
      f"all cited)")

Wrote /home/user/projects/shapley/reports/faithfulness_report.html (1503 KiB, 6 embedded figures, 22 references, all cited)


## BibTeX export

The same reference list as machine readable entries, written next to the
report as references.bib. The entry
count is asserted against the report's reference list so the two cannot
drift.

In [11]:
BIBTEX = r'''@inproceedings{lundberg2017unified,
  title = {A Unified Approach to Interpreting Model Predictions},
  author = {Lundberg, Scott M. and Lee, Su-In},
  booktitle = {Advances in Neural Information Processing Systems 30},
  year = {2017},
  eprint = {1705.07874},
  archiveprefix = {arXiv}
}

@inproceedings{ribeiro2016why,
  title = {"Why Should I Trust You?": Explaining the Predictions of Any Classifier},
  author = {Ribeiro, Marco Tulio and Singh, Sameer and Guestrin, Carlos},
  booktitle = {Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining},
  pages = {1135--1144},
  year = {2016},
  doi = {10.1145/2939672.2939778}
}

@inproceedings{zeiler2014visualizing,
  title = {Visualizing and Understanding Convolutional Networks},
  author = {Zeiler, Matthew D. and Fergus, Rob},
  booktitle = {European Conference on Computer Vision (ECCV)},
  year = {2014},
  doi = {10.1007/978-3-319-10590-1_53}
}

@article{doshivelez2017towards,
  title = {Towards A Rigorous Science of Interpretable Machine Learning},
  author = {Doshi-Velez, Finale and Kim, Been},
  journal = {arXiv preprint arXiv:1702.08608},
  year = {2017}
}

@inproceedings{yeh2019infidelity,
  title = {On the (In)fidelity and Sensitivity of Explanations},
  author = {Yeh, Chih-Kuan and Hsieh, Cheng-Yu and Suggala, Arun Sai and Inouye, David I. and Ravikumar, Pradeep},
  booktitle = {Advances in Neural Information Processing Systems 32},
  year = {2019},
  eprint = {1901.09392},
  archiveprefix = {arXiv}
}

@article{lage2019evaluation,
  title = {An Evaluation of the Human-Interpretability of Explanation},
  author = {Lage, Isaac and Chen, Emily and He, Jeffrey and Narayanan, Menaka and Kim, Been and Gershman, Samuel J. and Doshi-Velez, Finale},
  journal = {arXiv preprint arXiv:1902.00006},
  year = {2019}
}

@inproceedings{nguyen2018comparing,
  title = {Comparing Automatic and Human Evaluation of Local Explanations for Text Classification},
  author = {Nguyen, Dong},
  booktitle = {Proceedings of the 2018 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies},
  pages = {1069--1078},
  year = {2018},
  doi = {10.18653/v1/N18-1097}
}

@inproceedings{deyoung2020eraser,
  title = {{ERASER}: A Benchmark to Evaluate Rationalized {NLP} Models},
  author = {DeYoung, Jay and Jain, Sarthak and Rajani, Nazneen Fatema and Lehman, Eric and Xiong, Caiming and Socher, Richard and Wallace, Byron C.},
  booktitle = {Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics},
  pages = {4443--4458},
  year = {2020},
  doi = {10.18653/v1/2020.acl-main.408}
}

@inproceedings{jacovi2020towards,
  title = {Towards Faithfully Interpretable {NLP} Systems: How Should We Define and Evaluate Faithfulness?},
  author = {Jacovi, Alon and Goldberg, Yoav},
  booktitle = {Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics},
  pages = {4198--4205},
  year = {2020},
  doi = {10.18653/v1/2020.acl-main.386}
}

@inproceedings{atanasova2020diagnostic,
  title = {A Diagnostic Study of Explainability Techniques for Text Classification},
  author = {Atanasova, Pepa and Simonsen, Jakob Grue and Lioma, Christina and Augenstein, Isabelle},
  booktitle = {Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing},
  pages = {3256--3274},
  year = {2020},
  doi = {10.18653/v1/2020.emnlp-main.263}
}

@inproceedings{alvarezmelis2018robustness,
  title = {On the Robustness of Interpretability Methods},
  author = {Alvarez-Melis, David and Jaakkola, Tommi S.},
  booktitle = {ICML Workshop on Human Interpretability in Machine Learning},
  year = {2018},
  eprint = {1806.08049},
  archiveprefix = {arXiv}
}

@inproceedings{naylor2021quantifying,
  title = {Quantifying Explainability in {NLP} and Analyzing Algorithms for Performance-Explainability Tradeoff},
  author = {Naylor, Mitchell and French, Christi and Terker, Samantha and Kamath, Uday},
  booktitle = {ICML Workshop on Interpretable Machine Learning in Healthcare},
  year = {2021},
  eprint = {2107.05693},
  archiveprefix = {arXiv}
}

@inproceedings{bhatt2020evaluating,
  title = {Evaluating and Aggregating Feature-based Model Explanations},
  author = {Bhatt, Umang and Weller, Adrian and Moura, Jos{\'e} M. F.},
  booktitle = {Proceedings of the Twenty-Ninth International Joint Conference on Artificial Intelligence},
  year = {2020},
  eprint = {2005.00631},
  archiveprefix = {arXiv}
}

@inproceedings{hooker2019benchmark,
  title = {A Benchmark for Interpretability Methods in Deep Neural Networks},
  author = {Hooker, Sara and Erhan, Dumitru and Kindermans, Pieter-Jan and Kim, Been},
  booktitle = {Advances in Neural Information Processing Systems 32},
  pages = {9737--9748},
  year = {2019},
  eprint = {1806.10758},
  archiveprefix = {arXiv}
}

@inproceedings{adebayo2018sanity,
  title = {Sanity Checks for Saliency Maps},
  author = {Adebayo, Julius and Gilmer, Justin and Muelly, Michael and Goodfellow, Ian and Hardt, Moritz and Kim, Been},
  booktitle = {Advances in Neural Information Processing Systems 31},
  year = {2018},
  eprint = {1810.03292},
  archiveprefix = {arXiv}
}

@inproceedings{sundararajan2017axiomatic,
  title = {Axiomatic Attribution for Deep Networks},
  author = {Sundararajan, Mukund and Taly, Ankur and Yan, Qiqi},
  booktitle = {Proceedings of the 34th International Conference on Machine Learning},
  year = {2017},
  eprint = {1703.01365},
  archiveprefix = {arXiv}
}

@inproceedings{jain2019attention,
  title = {Attention is not Explanation},
  author = {Jain, Sarthak and Wallace, Byron C.},
  booktitle = {Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies},
  pages = {3543--3556},
  year = {2019},
  doi = {10.18653/v1/N19-1357}
}

@inproceedings{wiegreffe2019attention,
  title = {Attention is not not Explanation},
  author = {Wiegreffe, Sarah and Pinter, Yuval},
  booktitle = {Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th International Joint Conference on Natural Language Processing},
  pages = {11--20},
  year = {2019},
  doi = {10.18653/v1/D19-1002}
}

@inproceedings{bastings2020elephant,
  title = {The Elephant in the Interpretability Room: Why Use Attention as Explanation When We Have Saliency Methods?},
  author = {Bastings, Jasmijn and Filippova, Katja},
  booktitle = {Proceedings of the Third BlackboxNLP Workshop on Analyzing and Interpreting Neural Networks for NLP},
  pages = {149--155},
  year = {2020},
  doi = {10.18653/v1/2020.blackboxnlp-1.14}
}

@article{samek2017evaluating,
  title = {Evaluating the Visualization of What a Deep Neural Network Has Learned},
  author = {Samek, Wojciech and Binder, Alexander and Montavon, Gr{\'e}goire and Lapuschkin, Sebastian and M{\"u}ller, Klaus-Robert},
  journal = {IEEE Transactions on Neural Networks and Learning Systems},
  volume = {28},
  number = {11},
  pages = {2660--2673},
  year = {2017},
  doi = {10.1109/TNNLS.2016.2599820}
}

@inproceedings{dror2018hitchhikers,
  title = {The Hitchhiker's Guide to Testing Statistical Significance in Natural Language Processing},
  author = {Dror, Rotem and Baumer, Gili and Shlomov, Segev and Reichart, Roi},
  booktitle = {Proceedings of the 56th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)},
  pages = {1383--1392},
  year = {2018},
  doi = {10.18653/v1/P18-1128}
}

@article{zhou2021evaluating,
  title = {Evaluating the Quality of Machine Learning Explanations: A Survey on Methods and Metrics},
  author = {Zhou, Jianlong and Gandomi, Amir H. and Chen, Fang and Holzinger, Andreas},
  journal = {Electronics},
  volume = {10},
  number = {5},
  pages = {593},
  year = {2021},
  doi = {10.3390/electronics10050593}
}
'''

bibliography_path = REPORTS_DIRECTORY / "references.bib"
bibliography_path.write_text(BIBTEX)
entry_count = len(re.findall(r"^@", BIBTEX, flags=re.MULTILINE))
assert entry_count == len(REFERENCES), (
    f"{entry_count} BibTeX entries but {len(REFERENCES)} report references"
)
print(f"Wrote {bibliography_path} with {entry_count} entries, "
      "matching the report's reference list")

Wrote /home/user/projects/shapley/reports/references.bib with 22 entries, matching the report's reference list
